In [ ]:
# cell 1
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# cell 2
DATA_DIR = Path(r"..\Data\processed_data")

SURVEY_FILE = DATA_DIR / "cleaned_survey_base.csv"
POLICY_FILE = DATA_DIR / "cleaned_policy_epidemic.csv"

OUTPUT_MERGED = DATA_DIR / "data_merged.csv"
OUTPUT_ENCODED = DATA_DIR / "data_encoded.csv"

In [ ]:
# cell 3
survey_df = pd.read_csv(
    SURVEY_FILE,
    keep_default_na=False,
    low_memory=False
)

policy_df = pd.read_csv(
    POLICY_FILE,
    keep_default_na=False,
    low_memory=False
)

In [ ]:
# cell 4
# Convert date columns

survey_df["endtime"] = pd.to_datetime(
    survey_df["endtime"],
    errors="coerce"
)

policy_df["Date"] = pd.to_datetime(
    policy_df["Date"],
    errors="coerce"
)

In [ ]:
# cell 5
print("Survey date:", survey_df["endtime"].min(), "to", survey_df["endtime"].max())
print("Policy date:", policy_df["Date"].min(), "to", policy_df["Date"].max())

Survey date: 2020-06-24 00:00:00 to 2022-03-01 00:00:00
Policy date: 2020-04-02 00:00:00 to 2022-03-28 00:00:00


In [ ]:
# cell 6
# Check state names before merging

survey_states = sorted(survey_df["state"].dropna().unique())
policy_regions = sorted(policy_df["RegionName"].dropna().unique())

print("Survey states:")
print(survey_states)

print("\nPolicy regions:")
print(policy_regions)

print("\nStates in survey but not in policy:")
print(set(survey_states) - set(policy_regions))

print("\nRegions in policy but not in survey:")
print(set(policy_regions) - set(survey_states))

Survey states:
['Australian Capital Territory', 'New South Wales', 'Northern Territory', 'Queensland', 'South Australia', 'Tasmania', 'Victoria', 'Western Australia']

Policy regions:
['Australian Capital Territory', 'New South Wales', 'Northern Territory', 'Queensland', 'South Australia', 'Tasmania', 'Victoria', 'Western Australia']

States in survey but not in policy:
set()

Regions in policy but not in survey:
set()


In [ ]:
# cell 7
# Merge survey data with policy and epidemic data

merged_df = survey_df.merge(
    policy_df,
    how="left",
    left_on=["state", "endtime"],
    right_on=["RegionName", "Date"],
    indicator=True
)

print("Merged shape:", merged_df.shape)
print("Merge status:")
print(merged_df["_merge"].value_counts())

Merged shape: (41125, 77)
Merge status:
_merge
both          41125
left_only         0
right_only        0
Name: count, dtype: int64


In [ ]:
# cell 8
# Inspect unmatched survey records

unmatched_df = merged_df[merged_df["_merge"] != "both"].copy()

print("Number of unmatched survey rows:", unmatched_df.shape[0])

if unmatched_df.shape[0] > 0:
    print("Unmatched dates:")
    print(unmatched_df["endtime"].value_counts().sort_index())

    print("\nUnmatched states:")
    print(unmatched_df["state"].value_counts())

Number of unmatched survey rows: 0


In [ ]:
# cell 9
# Keep only successfully matched records

merged_df = merged_df[merged_df["_merge"] == "both"].copy()

merged_df = merged_df.drop(columns=["_merge"], errors="ignore")

print(merged_df.shape)

(41125, 76)


In [ ]:
# cell 10
# Drop duplicated merge key columns

merged_df = merged_df.drop(
    columns=["RegionName", "RegionCode", "Date"],
    errors="ignore"
)

print("Merged columns:")
print(merged_df.columns.tolist())
print(merged_df.shape)

Merged columns:
['RecordNo', 'endtime', 'i2_health', 'i9_health', 'i11_health', 'i12_health_1', 'i12_health_2', 'i12_health_3', 'i12_health_4', 'i12_health_5', 'i12_health_6', 'i12_health_7', 'i12_health_8', 'i12_health_11', 'i12_health_12', 'i12_health_13', 'i12_health_14', 'i12_health_15', 'i12_health_16', 'd1_health_1', 'd1_health_2', 'd1_health_3', 'd1_health_4', 'd1_health_5', 'd1_health_6', 'd1_health_7', 'd1_health_8', 'd1_health_9', 'd1_health_10', 'd1_health_11', 'd1_health_12', 'd1_health_13', 'd1_health_98', 'd1_health_99', 'age', 'gender', 'state', 'household_size', 'employment_status', 'WCRex2', 'cantril_ladder', 'PHQ4_1', 'PHQ4_2', 'PHQ4_3', 'PHQ4_4', 'WCRex1', 'i12_health_22', 'i12_health_23', 'i12_health_25', 'r1_1', 'r1_2', 'face_mask_valid_count', 'face_mask_behaviour_scale', 'face_mask_behaviour_binary', 'protective_behaviour_valid_count', 'protective_behaviour_scale', 'protective_behaviour_binary', 'social_avoidance_valid_count', 'social_avoidance_scale', 'social_av

In [ ]:
# cell 11
# Check missing values after merging

core_cols = [
    "endtime",
    "state",
    "face_mask_behaviour_binary",
    "protective_behaviour_binary",
    "social_avoidance_binary",
    "C1M_School closing",
    "C2M_Workplace closing",
    "H6M_Facial Coverings",
    "cases_14d_avg",
    "deaths_14d_avg"
]

core_cols = [col for col in core_cols if col in merged_df.columns]

core_missing = merged_df[core_cols].isna().sum()

print("Missing values in core columns:")
print(core_missing)

print("Total missing values in merged data:", merged_df.isna().sum().sum())

Missing values in core columns:
endtime                        0
state                          0
face_mask_behaviour_binary     0
protective_behaviour_binary    0
social_avoidance_binary        0
C1M_School closing             0
C2M_Workplace closing          0
H6M_Facial Coverings           0
cases_14d_avg                  0
deaths_14d_avg                 0
dtype: int64
Total missing values in merged data: 0


In [ ]:
# cell 12
# Save merged analysis base data

merged_df.to_csv(OUTPUT_MERGED, index=False)

In [ ]:
# cell 13
df = merged_df.replace("", np.nan)

missing_counts = df.isna().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)

print("Columns with missing values before cleaning:")
print(missing_counts)

Columns with missing values before cleaning:
household_size    989
daily_cases       597
daily_deaths        1
dtype: int64


In [ ]:
# cell 14
# Drop outcome-construction intermediate variables before encoding

outcome_construction_cols = [
    "face_mask_valid_count",
    "face_mask_behaviour_scale",
    "protective_behaviour_valid_count",
    "protective_behaviour_scale",
    "social_avoidance_valid_count",
    "social_avoidance_scale",
    "protective_behaviour_nomask_valid_count",
    "protective_behaviour_nomask_scale",
    "ConfirmedCases",
    "ConfirmedDeaths"
]

df = df.drop(columns=outcome_construction_cols, errors="ignore")

print(df.shape)

(41125, 63)


In [ ]:
# cell 15
# Remove rows with missing household_size

print("Shape before dropping missing household_size:", df.shape)

df = df.dropna(subset=["household_size"]).copy()

print(df.shape)

Shape before dropping missing household_size: (41125, 63)
(40136, 63)


In [ ]:
# cell 16
# Drop intermediate and raw columns not needed for encoded analysis data

# Raw comorbidity columns have already been summarized into d1_comorbidities
d1_raw_cols = [
    col for col in df.columns
    if col.startswith("d1_health_")
]

# Raw i12 behaviour columns have already been summarized into behavioural outcomes/scales
i12_raw_cols = [
    col for col in df.columns
    if col.startswith("i12_health_")
]

drop_cols = (
    d1_raw_cols +
    i12_raw_cols +
    [
        "daily_cases",
        "daily_deaths"
    ]
)

df = df.drop(columns=drop_cols, errors="ignore")

print(df.shape)
print("Dropped d1 columns:", len(d1_raw_cols))
print("Dropped i12 columns:", len(i12_raw_cols))

(40136, 29)
Dropped d1 columns: 15
Dropped i12 columns: 17


In [ ]:
# cell 17
# Convert numeric columns

numeric_cols = [
    "RecordNo",
    "i2_health",
    "age",
    "household_size",
    "cantril_ladder",
    "r1_1",
    "r1_2",
    "face_mask_valid_count",
    "face_mask_behaviour_scale",
    "protective_behaviour_valid_count",
    "protective_behaviour_scale",
    "social_avoidance_valid_count",
    "social_avoidance_scale",
    "protective_behaviour_nomask_valid_count",
    "protective_behaviour_nomask_scale",
    "week_number",
    "C1M_School closing",
    "C2M_Workplace closing",
    "H6M_Facial Coverings",
    "ConfirmedCases",
    "ConfirmedDeaths",
    "cases_14d_avg",
    "deaths_14d_avg"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
# cell 18
# Check cleaned merged data before dummy encoding

missing_counts_after = df.isna().sum()
missing_counts_after = missing_counts_after[missing_counts_after > 0].sort_values(ascending=False)

print(df.shape)
print("Remaining missing values:")
print(missing_counts_after)

(40136, 29)
Remaining missing values:
Series([], dtype: int64)


In [ ]:
# cell 19
# Encode binary outcomes

outcome_cols = [
    "face_mask_behaviour_binary",
    "protective_behaviour_binary",
    "social_avoidance_binary"
]

binary_map = {
    "No": 0,
    "Yes": 1
}

for col in outcome_cols:
    if col in df.columns:
        df[col] = df[col].map(binary_map)

print("Outcome distributions:")
for col in outcome_cols:
    if col in df.columns:
        print(f"\n{col}")
        print(df[col].value_counts(dropna=False))

Outcome distributions:

face_mask_behaviour_binary
face_mask_behaviour_binary
1    21658
0    18478
Name: count, dtype: int64

protective_behaviour_binary
protective_behaviour_binary
1    26149
0    13987
Name: count, dtype: int64

social_avoidance_binary
social_avoidance_binary
1    23699
0    16437
Name: count, dtype: int64


In [ ]:
# cell 20
# Dummy encode categorical variables

encoded_df = df.copy()

exclude_from_dummy = [
    "endtime"
]

categorical_cols = [
    col for col in encoded_df.select_dtypes(include=["object"]).columns
    if col not in exclude_from_dummy
]

print("Categorical columns to encode:")
print(categorical_cols)

encoded_df = pd.get_dummies(
    encoded_df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

print(encoded_df.shape)

Categorical columns to encode:
['i9_health', 'i11_health', 'gender', 'state', 'employment_status', 'WCRex2', 'PHQ4_1', 'PHQ4_2', 'PHQ4_3', 'PHQ4_4', 'WCRex1', 'd1_comorbidities']
(40136, 67)


In [ ]:
# cell 21
# Check non-numeric columns after encoding

non_numeric_cols = encoded_df.select_dtypes(
    exclude=["number", "datetime64[ns]"]
).columns.tolist()

print("Non-numeric columns after encoding:")
print(non_numeric_cols)

Non-numeric columns after encoding:
[]


In [ ]:
# cell 22
# Save encoded data

encoded_df.to_csv(OUTPUT_ENCODED, index=False)

print(encoded_df.shape)

(40136, 67)


In [ ]:
# cell 23
# Final checks

print("Cleaned merged shape:", df.shape)
print("Encoded shape:", encoded_df.shape)

print("\nRemaining missing values in encoded data:")
missing_encoded = encoded_df.isna().sum()
print(missing_encoded[missing_encoded > 0].sort_values(ascending=False))

print("\nOutcome distributions:")
for col in outcome_cols:
    if col in encoded_df.columns:
        print(f"\n{col}")
        print(encoded_df[col].value_counts(dropna=False))

print("\nPolicy variable distributions:")
policy_cols = [
    "C1M_School closing",
    "C2M_Workplace closing",
    "H6M_Facial Coverings"
]

for col in policy_cols:
    if col in encoded_df.columns:
        print(f"\n{col}")
        print(encoded_df[col].value_counts().sort_index())

print("\nEpidemic severity summary:")
epidemic_cols = [
    "cases_14d_avg",
    "deaths_14d_avg"
]

print(encoded_df[epidemic_cols].describe())

Cleaned merged shape: (40136, 29)
Encoded shape: (40136, 67)

Remaining missing values in encoded data:
Series([], dtype: int64)

Outcome distributions:

face_mask_behaviour_binary
face_mask_behaviour_binary
1    21658
0    18478
Name: count, dtype: int64

protective_behaviour_binary
protective_behaviour_binary
1    26149
0    13987
Name: count, dtype: int64

social_avoidance_binary
social_avoidance_binary
1    23699
0    16437
Name: count, dtype: int64

Policy variable distributions:

C1M_School closing
C1M_School closing
0.0     2176
1.0    30679
2.0     1888
3.0     5393
Name: count, dtype: int64

C2M_Workplace closing
C2M_Workplace closing
0.0     1575
1.0    30340
2.0     2487
3.0     5734
Name: count, dtype: int64

H6M_Facial Coverings
H6M_Facial Coverings
0.0     4926
1.0     5574
2.0    17596
3.0     5857
4.0     6183
Name: count, dtype: int64

Epidemic severity summary:
       cases_14d_avg  deaths_14d_avg
count   40136.000000    40136.000000
mean     1399.777645        2.1734